# Supplementary Results 5 — Importance of the secondary fine-mapping signals

Within a disease-associated region, does the most significant credible set carry the functional
evidence, or do the secondary ones? Qualifying disease credible sets only, SuSiE and SuSiE-inf
only; within each region and study the credible sets are ranked by P value, the most significant
counting as primary and the rest as secondary.

Numbers are written to `results/sr05_secondary_signals.json`.

**Provenance.** GAPS.md recorded this section as having no code anywhere. It does:
`~/Projects/EGL_and_training_set/archive/gentropy_paper/12_importnace_of_secondary_signals.ipynb`,
whose `get_feature_numbers()` is ported here. Its input `max_feature_per_CS.parquet` was the maximum
of every numeric feature-matrix column per credible set over protein-coding genes
(`01d_the_max_feature_value_per_CS.ipynb`), rebuilt below.

**One definition was changed on purpose.** The archive computes the rescued share as *every credible
set carrying evidence in the bare regions, divided by the number of bare regions* — 2,333 / 2,862 =
81.5%, the published value. Numerator and denominator are different units (credible sets against
regions), the numerator includes primary credible sets belonging to other studies of the same region,
and the quotient can exceed 100%. The section and the main text both claim a **share of regions**, so
that is what is registered here: **1,451 of 2,862 regions, 50.7%**, have a secondary credible set
carrying at least one feature. Every reading is printed side by side below, the published one
included, and `tools/expected_numbers.tsv` keeps 81.5 until the manuscript text is changed.

In [1]:
import pandas as pd
from gentropy.common.session import Session
from pyspark.sql import Window
from pyspark.sql import functions as f

from manuscript_methods import paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

CLPP, H4, VEP_PAV = 0.01, 0.8, 0.66

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 15:34:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## The credible sets in scope

Qualifying disease credible sets fine-mapped by SuSiE or SuSiE-inf; PICS ignores secondary signals
by construction and is excluded.

In [2]:
qualifying = session.spark.read.parquet(paper.derived("qualifying_credible_sets")).select("studyLocusId")
cs = (
    session.spark.read.parquet(paper.release("credible_set"))
    .join(qualifying, "studyLocusId", "inner")
    .select("studyLocusId", "studyId", "region", "finemappingMethod", "pValueMantissa", "pValueExponent")
    .cache()
)
print(cs.groupBy("finemappingMethod").count().toPandas().to_string(index=False))

susie = cs.filter(f.col("finemappingMethod").isin("SuSie", "SuSiE-inf")).cache()
print(f"SuSiE credible sets: {susie.count():,}")

finemappingMethod  count
            SuSie  15075
        SuSiE-inf  27561
             PICS  27982


SuSiE credible sets: 42,636


In [3]:
# The maximum of each evidence feature over the protein-coding genes of a credible set.
feature_max = (
    session.spark.read.parquet(paper.release("l2g_feature_matrix"))
    .filter(f.col("isProteinCoding") == 1)
    .groupBy("studyLocusId")
    .agg(
        f.max("eQtlColocClppMaximum").alias("eQtlColocClppMaximum"),
        f.max("eQtlColocH4Maximum").alias("eQtlColocH4Maximum"),
        f.max("pQtlColocClppMaximum").alias("pQtlColocClppMaximum"),
        f.max("pQtlColocH4Maximum").alias("pQtlColocH4Maximum"),
        f.max("vepMaximum").alias("vepMaximum"),
    )
)
annotated = susie.join(feature_max, "studyLocusId", "inner").cache()
print(f"credible sets with a feature row: {annotated.count():,}")

credible sets with a feature row: 42,081


## Primary against secondary signals

`get_feature_numbers()` from the archive notebook, unchanged: rank within `(region, studyId)` by
P value, keep the regions that have a secondary signal, and compare the two groups.

In [4]:
HAS_EQTL = (f.col("eQtlColocClppMaximum") >= CLPP) | (f.col("eQtlColocH4Maximum") >= H4)
HAS_PQTL = (f.col("pQtlColocClppMaximum") >= CLPP) | (f.col("pQtlColocH4Maximum") >= H4)
HAS_PAV = f.col("vepMaximum") >= VEP_PAV
HAS_ANY = HAS_EQTL | HAS_PQTL | HAS_PAV


def feature_numbers(frame, label):
    """Every number Supplementary Results 5 reports, for one set of credible sets."""
    window = Window.partitionBy("region", "studyId").orderBy(
        (f.col("pValueMantissa") * (10 ** f.col("pValueExponent"))).asc()
    )
    ranked = frame.withColumn("regionRank", f.row_number().over(window)).cache()
    regions = ranked.select("region").distinct().count()

    with_secondary = ranked.filter(f.col("regionRank") > 1).select("region").distinct().cache()
    in_scope = ranked.join(with_secondary, "region", "inner").cache()
    primary = in_scope.filter(f.col("regionRank") == 1).cache()
    secondary = in_scope.filter(f.col("regionRank") > 1).cache()

    # Regions whose primary credible set carries none of the three, and how many of them a secondary
    # signal rescues. `rescued` counts regions, which is what the claim is about; `archive_ratio` is
    # the credible-set-over-region quotient the published 81.5% came from, kept for comparison.
    bare = primary.filter(~HAS_ANY).select("region").distinct().cache()
    n_bare = bare.count()
    rescued = secondary.join(bare, "region", "inner").filter(HAS_ANY).select("region").distinct().count()
    rescued_any_cs = in_scope.join(bare, "region", "inner").filter(HAS_ANY).select("region").distinct().count()
    archive_ratio = in_scope.join(bare, "region", "inner").filter(HAS_ANY).count()

    n_primary, n_secondary = primary.count(), secondary.count()
    return {
        "set": label,
        "regions": regions,
        "regions with a secondary signal": with_secondary.count(),
        "primary CSs": n_primary,
        "secondary CSs": n_secondary,
        "regions whose primary carries nothing": n_bare,
        "of those, rescued by a secondary (%)": round(100 * rescued / n_bare, 1),
        "of those, any CS carries a feature (%)": round(100 * rescued_any_cs / n_bare, 1),
        "archive ratio, CSs with a feature over regions (%)": round(100 * archive_ratio / n_bare, 1),
        "primary with an eQTL (%)": round(100 * primary.filter(HAS_EQTL).count() / n_primary, 1),
        "secondary with an eQTL (%)": round(100 * secondary.filter(HAS_EQTL).count() / n_secondary, 1),
        "primary with a pQTL (%)": round(100 * primary.filter(HAS_PQTL).count() / n_primary, 1),
        "secondary with a pQTL (%)": round(100 * secondary.filter(HAS_PQTL).count() / n_secondary, 1),
        "primary with a PAV (%)": round(100 * primary.filter(HAS_PAV).count() / n_primary, 1),
        "secondary with a PAV (%)": round(100 * secondary.filter(HAS_PAV).count() / n_secondary, 1),
    }


all_sets = feature_numbers(annotated, "all qualifying disease CSs")
pd.Series(all_sets).to_frame("value")

,value
set,all qualifying disease CSs
regions,24558
regions with a secondary signal,6354
primary CSs,8780
secondary CSs,11439
regions whose primary carries nothing,2862
"of those, rescued by a secondary (%)",50.7
"of those, any CS carries a feature (%)",52.1
"archive ratio, CSs with a feature over regions (%)",81.5
primary with an eQTL (%),48.5


In [5]:
numbers["S5.01"] = all_sets["regions"]
numbers["S5.02"] = all_sets["regions with a secondary signal"]
numbers["S5.03"] = all_sets["primary CSs"]
numbers["S5.04"] = all_sets["secondary CSs"]
numbers["S5.05"] = all_sets["regions whose primary carries nothing"]
numbers["S5.06"] = all_sets["of those, rescued by a secondary (%)"]
numbers["S5.07"] = all_sets["primary with an eQTL (%)"]
numbers["S5.08"] = all_sets["secondary with an eQTL (%)"]
numbers["S5.09"] = all_sets["primary with a pQTL (%)"]
numbers["S5.10"] = all_sets["secondary with a pQTL (%)"]
numbers["S5.11"] = all_sets["primary with a PAV (%)"]
numbers["S5.12"] = all_sets["secondary with a PAV (%)"]
# Results 3 quotes the same share (R3.12); this is the only place it is computed. Both
# carry the recomputed share of regions, not the archive's credible-set-over-region ratio.
numbers["R3.12"] = numbers["S5.06"]
print(numbers)

{'S5.01': 24558, 'S5.02': 6354, 'S5.03': 8780, 'S5.04': 11439, 'S5.05': 2862, 'S5.06': 50.7, 'S5.07': 48.5, 'S5.08': 42.4, 'S5.09': 9.8, 'S5.10': 7.2, 'S5.11': 17.5, 'S5.12': 14.6, 'R3.12': 50.7}


## The same restricted to replicated credible sets

"This pattern was consistent even when the analysis was restricted to replicated CSs only."

In [6]:
replicated = session.spark.read.parquet(paper.derived("replicated_gwas_cs")).select("studyLocusId")
replicated_only = feature_numbers(annotated.join(replicated, "studyLocusId", "inner"), "replicated only")
pd.DataFrame([all_sets, replicated_only]).set_index("set").T

set,all qualifying disease CSs,replicated only
regions,24558.0,8470.0
regions with a secondary signal,6354.0,1168.0
primary CSs,8780.0,1644.0
secondary CSs,11439.0,1839.0
regions whose primary carries nothing,2862.0,591.0
"of those, rescued by a secondary (%)",50.7,42.5
"of those, any CS carries a feature (%)",52.1,46.0
"archive ratio, CSs with a feature over regions (%)",81.5,73.8
primary with an eQTL (%),48.5,37.3
secondary with an eQTL (%),42.4,33.9


## Write the results

In [7]:
print(paper.save_results("sr05_secondary_signals", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr05_secondary_signals.json


,computed
S5.01,24558.0
S5.02,6354.0
S5.03,8780.0
S5.04,11439.0
S5.05,2862.0
S5.06,50.7
S5.07,48.5
S5.08,42.4
S5.09,9.8
S5.10,7.2
